In [11]:
import json
import os
import sys
from pathlib import Path

os.environ["MESA_DIR"] = "/Users/mchau/mesa"   # Replace if your MESA is elsewhere

print("MESA_DIR =", os.environ["MESA_DIR"])

def locate_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'tutorial' / 'inlist_project').is_file():
            return candidate
    raise FileNotFoundError('Could not locate tutorial/inlist_project')

PROJECT_ROOT = locate_project_root()
PIPELINE_DIR = PROJECT_ROOT / 'scripts' / 'mams_profiles' / 'pipeline'
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from mams_pipeline import (
    build_configs, inspect_runs, model_label, prepare_grid,
    render_inlist_project, run_local, submit_slurm,
)

print(f'Project root: {PROJECT_ROOT}')

# Main user-editable parameters
MASSES_MSUN = [5]
INITIAL_Y = 0.2703
INITIAL_Z = 0.0142
STAGE = "RGB"
CENTRAL_H1_LIMIT = 0.01
RGB_RADIUS_LIMIT = 0.5


# Slurm settings used only if SUBMIT_SLURM is enabled.
PARTITION = 'typhon'
WALLTIME = '7-00:00:00'
CPUS_PER_TASK = 1

# An environment override makes validation and alternate output locations easy.
OUTPUT_ROOT = Path(os.environ.get(
    'MAMS_OUTPUT_ROOT', PROJECT_ROOT / 'results' / 'mams_profiles'
)).resolve()
TEMPLATE_DIR = PROJECT_ROOT / 'tutorial'
OVERWRITE = False
PREPARE_RUNS = True
RUN_LOCAL = True       # Runs models sequentially and waits; potentially slow.
SUBMIT_SLURM = False    # Submits all prepared models and returns job IDs.

if RUN_LOCAL and SUBMIT_SLURM:
    raise ValueError('Choose local execution or Slurm submission, not both.')


configs = build_configs(
    MASSES_MSUN,
    initial_y=INITIAL_Y,
    initial_z=INITIAL_Z,
    central_h1_limit=CENTRAL_H1_LIMIT,
    stage=STAGE,
    rgb_radius_limit=RGB_RADIUS_LIMIT,
    partition=PARTITION,
    walltime=WALLTIME,
    cpus_per_task=CPUS_PER_TASK,
)

for config in configs:
    print(f'{config.mass_msun:8g} M_sun  ->  {model_label(config.stage, config.mass_msun, config.initial_z)}')

template_text = (TEMPLATE_DIR / 'inlist_project').read_text()
preview = render_inlist_project(template_text, configs[0])
keys_to_show = (
    'filename_for_profile_when_terminate', 'save_pulse_data_filename',
    'initial_mass', 'initial_y', 'initial_z', 'Zbase',
    'xa_central_lower_limit(1)', 'rgb_radius_limit',
)
for line in preview.splitlines():
    if any(key in line for key in keys_to_show):
        print(line)

assert f"'{model_label(configs[0].stage, configs[0].mass_msun, configs[0].initial_z)}.profile'" in preview
assert f"'{model_label(configs[0].stage, configs[0].mass_msun, configs[0].initial_z)}.mesa'" in preview
print('In-memory smoke test passed.')

runs = []
if PREPARE_RUNS:
    runs = prepare_grid(
        configs, TEMPLATE_DIR, OUTPUT_ROOT, overwrite=OVERWRITE
    )
    print(f'Prepared {len(runs)} run(s) under {OUTPUT_ROOT}')
else:
    print('PREPARE_RUNS is false; no directories were written.')

inspect_runs(runs)

job_ids = {}
if RUN_LOCAL:
    for run in runs:
        print(f'Running {run.label} ...', flush=True)
        run_local(run)
elif SUBMIT_SLURM:
    for run in runs:
        job_ids[run.label] = submit_slurm(run)
    print(json.dumps(job_ids, indent=2))
else:
    print('Setup only: no MESA processes or Slurm jobs were started.')

status = inspect_runs(runs)
print(json.dumps(status, indent=2))

if runs:
    metadata = json.loads(runs[0].metadata_path.read_text())
    print('\nFirst run metadata:')
    print(json.dumps(metadata, indent=2))

    if runs[0].gyre_path.is_file():
        fields = runs[0].gyre_path.read_text().splitlines()[0].split()
        names = ['zones', 'mass_g', 'radius_cm', 'luminosity_erg_s', 'version']
        print('\nGYRE header:', dict(zip(names, fields)))

MESA_DIR = /Users/mchau/mesa
Project root: /Users/mchau/mesa_tutorial
       5 M_sun  ->  RGB5Msun_Z0p0142
   filename_for_profile_when_terminate = 'RGB5Msun_Z0p0142.profile'
   save_pulse_data_filename = 'RGB5Msun_Z0p0142.mesa'
   initial_mass = 5
   initial_y = 0.2703  ! Helium mass fraction
   initial_z = 0.0142  ! Metal mass fraction, hydrogen = 1 - metal - helium
   xa_central_lower_limit(1) = 0.0
  use_Zbase_for_Type1 = .true.
  Zbase = 0.0142
In-memory smoke test passed.
Prepared 1 run(s) under /Users/mchau/mesa_tutorial/results/mams_profiles
Running RGB5Msun_Z0p0142 ...
[
  {
    "label": "RGB5Msun_Z0p0142",
    "prepared": true,
    "mesa_profile": true,
    "gyre_model": true,
    "run_dir": "/Users/mchau/mesa_tutorial/results/mams_profiles/RGB5Msun_Z0p0142"
  }
]

First run metadata:
{
  "config": {
    "central_h1_limit": 0.01,
    "cpus_per_task": 1,
    "initial_y": 0.2703,
    "initial_z": 0.0142,
    "mass_msun": 5,
    "partition": "typhon",
    "rgb_radius_limit": 0.5

In [18]:
from pathlib import Path

run_dir = Path(
    "/Users/mchau/mesa_tutorial/results/mams_profiles/"
    "RGB5Msun_Z0p0142_logR0p5"
)

print("RUN DIR:", run_dir)
print()

for f in ["inlist", "inlist_project"]:
    path = run_dir / f
    print(f"{f}: exists =", path.exists())
    if path.exists():
        for line in path.read_text().splitlines():
            if "log_Rsurf_upper_limit" in line:
                print("  ", repr(line))

RUN DIR: /Users/mchau/mesa_tutorial/results/mams_profiles/RGB5Msun_Z0p0142_logR0p5

inlist: exists = True
inlist_project: exists = True
   '   log_Rsurf_upper_limit = 0.5'


In [19]:
from pathlib import Path

run_dir = Path(
    "/Users/mchau/mesa_tutorial/results/mams_profiles/"
    "RGB5Msun_Z0p0142_logR0p5"
)

history = run_dir / "LOGS" / "history.data"

print("Exists:", history.exists())
print("Path:", history)

Exists: True
Path: /Users/mchau/mesa_tutorial/results/mams_profiles/RGB5Msun_Z0p0142_logR0p5/LOGS/history.data


In [20]:
from pathlib import Path
import numpy as np

run_dir = Path(
    "/Users/mchau/mesa_tutorial/results/mams_profiles/"
    "RGB5Msun_Z0p0142"
)

history_path = run_dir / "LOGS" / "history.data"

print("History exists:", history_path.exists())
print("Path:", history_path)

# Find the header line containing the column names
lines = history_path.read_text().splitlines()

for i, line in enumerate(lines):
    if "model_number" in line and "star_age" in line:
        header_line = i
        break

print("Header line:", header_line)
print(lines[header_line])

History exists: True
Path: /Users/mchau/mesa_tutorial/results/mams_profiles/RGB5Msun_Z0p0142/LOGS/history.data
Header line: 5
                            model_number                                num_zones                                 star_age                                   log_dt                                star_mass                               log_xmstar                             log_abs_mdot                           mass_conv_core                             conv_mx1_top                             conv_mx1_bot                             conv_mx2_top                             conv_mx2_bot                                  mx1_top                                  mx1_bot                                  mx2_top                                  mx2_bot                                   log_LH                                  log_LHe                                   log_LZ                                 log_Lnuc                                       pp              

In [22]:
import pandas as pd

df = pd.read_csv(
    history_path,
    sep=r"\s+",
    skiprows=header_line + 1,
)

print(df.columns.tolist())

['1', '560', '1.0000000000000001E-005', '-4.9999999999999991E+000', '5.0000000000000000E+000', '3.3997475914696686E+001', '-9.8999999999999986E+001', '0.0000000000000000E+000', '9.9999981187905085E-001', '2.3748116375852248E-002', '0.0000000000000000E+000.1', '0.0000000000000000E+000.2', '9.9999981187905085E-001.1', '2.3748116375852248E-002.1', '0.0000000000000000E+000.3', '0.0000000000000000E+000.4', '-1.2330290385337936E+001', '-9.8999999999999986E+001.1', '-9.8999999999999986E+001.2', '-1.2330290385337936E+001.1', '-1.2330290385337936E+001.2', '-3.6045094134397353E+001', '-9.8999999999999986E+001.3', '-2.0000000000000000E+001', '-2.0000000000000000E+001.1', '-2.0000000000000000E+001.2', '-2.0000000000000000E+001.3', '-2.0000000000000000E+001.4', '-2.0000000000000000E+001.5', '-2.0000000000000000E+001.6', '-2.0000000000000000E+001.7', '0.0000000000000000E+000.5', '0.0000000000000000E+000.6', '0.0000000000000000E+000.7', '0.0000000000000000E+000.8', '0.0000000000000000E+000.9', '3.673

In [26]:
import pandas as pd

df = pd.read_csv(
    history_path,
    sep=r"\s+",
    header=header_line,
)

print(df.columns.tolist())
print(df.head())

['1', '560', '1.0000000000000001E-005', '-4.9999999999999991E+000', '5.0000000000000000E+000', '3.3997475914696686E+001', '-9.8999999999999986E+001', '0.0000000000000000E+000', '9.9999981187905085E-001', '2.3748116375852248E-002', '0.0000000000000000E+000.1', '0.0000000000000000E+000.2', '9.9999981187905085E-001.1', '2.3748116375852248E-002.1', '0.0000000000000000E+000.3', '0.0000000000000000E+000.4', '-1.2330290385337936E+001', '-9.8999999999999986E+001.1', '-9.8999999999999986E+001.2', '-1.2330290385337936E+001.1', '-1.2330290385337936E+001.2', '-3.6045094134397353E+001', '-9.8999999999999986E+001.3', '-2.0000000000000000E+001', '-2.0000000000000000E+001.1', '-2.0000000000000000E+001.2', '-2.0000000000000000E+001.3', '-2.0000000000000000E+001.4', '-2.0000000000000000E+001.5', '-2.0000000000000000E+001.6', '-2.0000000000000000E+001.7', '0.0000000000000000E+000.5', '0.0000000000000000E+000.6', '0.0000000000000000E+000.7', '0.0000000000000000E+000.8', '0.0000000000000000E+000.9', '3.673

In [34]:
from pathlib import Path

run_dir = Path(
    "/Users/mchau/mesa_tutorial/results/mams_profiles/"
    "RGB5Msun_Z0p0142"
)

out_path = run_dir / "mesa.out"

print("Exists:", out_path.exists())
print("Size:", out_path.stat().st_size, "bytes")

text = out_path.read_text()

print(text[-15000:])

Exists: True
Size: 22028 bytes
5.478219   4041.198 -22.232901 -22.232901   5.000000   5.000000   0.715500   0.000715   0.270300 -11.096374    482      0
-5.0000E+00   5.477676   2.116577 -99.000000 -22.876908 -99.000000   0.000000   0.270300   0.006643   0.014200   0.025275      2
 1.1000E-03  -4.627722   3.613889 -99.000000 -99.000000  -8.383153   0.000000   0.002418   0.001366   0.000575  0.000E+00        max_dt

        120   5.478219   4033.173 -22.232900 -22.232900   5.000000   5.000000   0.715500   0.000715   0.270300 -11.096372    482      0
-5.0000E+00   5.477676   2.116550 -99.000000 -22.876907 -99.000000   0.000000   0.270300   0.006643   0.014200   0.025275      2
 1.2000E-03  -4.627722   3.610381 -99.000000 -99.000000  -8.380883   0.000000   0.002418   0.001366   0.000575  0.000E+00        max_dt

        130   5.478220   4026.077 -22.232899 -22.232899   5.000000   5.000000   0.715500   0.000715   0.270300 -11.096369    483      0
-5.0000E+00   5.477676   2.116525 -99.00000